# Assignment 2 — Transfer Learning (BERT on TweetEval–Irony)

### Keanu De Cleene, kdec819, 605624678

In this assignment, you will fine-tune a pre-trained **BERT** model on the **TweetEval** *irony* subset and explore strategies to achieve strong performance under different training regimes.

- **Model:** [bert-base-uncased](https://huggingface.co/bert-base-uncased)  
- **Dataset:** [TweetEval](https://huggingface.co/datasets/cardiffnlp/tweet_eval/viewer/irony) → subset **`irony`** (binary labels: `non_irony`, `irony`). Use the provided `train/validation/test` splits and the dataset’s class mapping.

### Tasks
1. **Load & Inspect the Data**  
   Report split sizes, class distribution, and text length statistics (e.g., word/character histograms). Include at least one plot.
2. **Tokenization**  
   Apply the BERT tokenizer with justified `max_seq_length`; enable truncation/padding; fix random seeds for reproducibility.
3. **Standard Fine-Tuning**  
   Fine-tune `bert-base-uncased` with a small hyperparameter grid (e.g., batch size, learning rate, max_seq_length). Log training curves and select the best configuration by **validation accuracy**.
4. **Error Analysis**  
   You need to provide insights on the model's wrong predictions from each category.
5. **Fine-tuning with Limited Training Data**  
   Re-run training with only a small fraction of the training data and explore techniques to recover performance.

**Primary metric:** test **accuracy**.  
**Additional reporting:** macro-F1, confusion matrix, and brief error analysis with misclassified examples (predicted vs. gold, with short diagnoses).


 ## Task 1: Load and Inspect the Data

### Task 1.1 — Define model & dataset names (given)
- **Model:** [`bert-base-uncased`](https://huggingface.co/bert-base-uncased)  
- **Dataset:** [`cardiffnlp/tweet_eval`](https://huggingface.co/datasets/cardiffnlp/tweet_eval), subset **`irony`** (binary: `non_irony`, `irony`)

> You will re-use these identifiers throughout the notebook.

---

### Task 1.2 — Load dataset & build label mappings
- Load the **train/validation/test** splits for the **`irony`** subset using `datasets.load_dataset`.
- Extract the label names from `ds["train"].features["label"].names`.
- Build **`id2label`** and **`label2id`** dictionaries for consistency with Hugging Face `AutoModelForSequenceClassification`.

**Expected artifacts**
- Printed split sizes (e.g., `{'train': N1, 'validation': N2, 'test': N3}`).
- Printed label names and a `dict` view of `id2label` / `label2id`.

---

### Task 1.3 — Plot statistical information
Produce plots and summaries that help you understand the dataset:

1. **Class distribution** (per split: train/validation/test)  
   - Bar chart with counts for `non_irony` vs. `irony` per split.
2. **Tweet length distributions** (train split)  
   - Histograms for:
     - **Character lengths** (e.g., 50 bins)
     - **Word counts** (e.g., whitespace tokenization as a proxy, still use 50 as bin size)  
   - Report key stats: mean, median, and 95th percentile for both character and word lengths.

**Notes**
- Keep axes labeled (units: “chars”, “words”) and add informative titles.
- Save or show figures inline.
- These stats will justify later choices for `max_seq_length` (e.g., 64/96/128).


In [1]:
import os
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

# Reproducibility
GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

MODEL_NAME = "bert-base-uncased"
DATASET_NAME = "cardiffnlp/tweet_eval"
SUBSET_NAME = "irony"

c:\Users\keanu\anaconda3\envs\compsci714\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\keanu\anaconda3\envs\compsci714\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [ ]:
# ============================================
# Task Procedure
# ============================================
# Goal: Load the TweetEval–irony dataset, build label mappings, inspect split sizes
#       and basic length statistics, then visualize class distribution and tweet
#       length histograms. Use the outputs to justify a reasonable max_seq_length.
#
# 1) Load dataset splits
#    - Use: datasets.load_dataset(DATASET_NAME, SUBSET_NAME)
#    - Expect three splits: 'train', 'validation', 'test'
#    - Immediately print the split sizes with: {split_name: len(split)}




#
# 2) Extract label names and build mappings
#    - From ds["train"].features["label"].names obtain label_names
#    - Build:
#        id2label = {i: name for i, name in enumerate(label_names)}
#        label2id = {name: i for i, name in enumerate(label_names)}
#    - Print label_names and id2label for sanity check
#
# 3) Prepare text fields per split
#    - Extract lists:
#        train_texts = ds["train"]["text"]
#        val_texts   = ds["validation"]["text"]
#        test_texts  = ds["test"]["text"]
#    - No modifications to text content yet (keep raw tweets)
#
# 4) Define helper to compute lengths
#    - lengths(texts) should return:
#        - char_lens: number of characters per tweet (len(t))
#        - word_lens: approximate token count per tweet (len(t.split()))
#    - Return numpy arrays for downstream stats and plotting
#
# 5) Compute length arrays for each split
#    - For train/validation/test texts separately:
#        train_char, train_word = lengths(train_texts)
#        val_char,   val_word   = lengths(val_texts)
#        test_char,  test_word  = lengths(test_texts)
#
# 6) Summarize train length statistics
#    - Print for characters and words on the TRAIN split:
#        - mean (use .mean())
#        - median (np.median)
#        - 95th percentile (np.percentile(..., 95))
#    - These numbers will inform the tokenizer max_seq_length candidates later
#
# 7) Compute class distribution per split
#    - Implement label_hist(split):
#        * Read split["label"]
#        * For each class index i in range(len(label_names)), count occurrences
#    - Get per-split counts:
#        train_counts = label_hist(ds["train"])
#        val_counts   = label_hist(ds["validation"])
#        test_counts  = label_hist(ds["test"])
#    - Print at least train/test class-count dicts for quick inspection
#
# 8) Plot class distribution (bar chart)
#    - Create a grouped bar chart with bars for train/validation/test on the same x-axis
#      (x positions for classes; width ~0.25; legend for splits)
#    - Set xticks to label_names
#    - Add title: "Class distribution by split"
#    - Add y-axis label: "count"
#    - Show the figure
#
# 9) Plot tweet length histograms on TRAIN split
#    - Two separate histograms (NO subplots):
#        (a) Characters: histogram of train_char with bins=50
#            * Title: "Train tweet lengths (chars)"
#            * x-label: "chars", y-label: "count"
#        (b) Words: histogram of train_word with bins=50
#            * Title: "Train tweet lengths (words)"
#            * x-label: "words", y-label: "count"
#    - Show each figure after configuration
#
# 10) Notes / Best Practices
#    - Keep axes labels and informative titles on all plots.
#    - Do not modify text content (no preprocessing yet) to avoid biasing stats.
#    - If class imbalance is evident from the bar chart, note it for later (e.g., F1 reporting, sampling).
#    - Use these length stats to propose a justified max_seq_length grid
#      (e.g., around the 95th percentile of word or subword lengths).
#
#
# Deliverables for this step
#    - Printed: split sizes, label_names, id2label mapping
#    - Printed: train length stats (chars & words: mean/median/p95)
#    - Plots: class distribution (train/val/test), histograms of train char/word lengths
#    - Written justification (in your report/markdown): chosen max_seq_length candidates
#      for downstream tokenization based on the observed distributions.


## Task 2: Tokenization

- **Define** a tokenization function and, if needed, a helper function to tokenize the dataset.
- **Do not** hard-code `max_length`; treat it as an adjustable hyperparameter for training. The range of `max_length` can be dertmined based on your insights on data analysis above.
- If possible, you can define the following two functions here (reference only):
  1. A factory function `make_tokenize_fn(max_len)` that returns a callable to tokenize batches with the given `max_len`.
  2. A helper `prepare_datasets_for_max_len(ds, max_len)` that applies the tokenizer to all splits and prepares tensors.


In [9]:
# ============================================
# Task Procedure: Tokenization & Dataset Prep
# ============================================
# Goal: Create a reusable tokenization pipeline that (a) instantiates the
#       pretrained tokenizer, (b) provides a factory to tokenize with a chosen
#       max_len, (c) uses dynamic padding via a data collator, and (d) produces
#       tensor-ready DatasetDict splits compatible with Hugging Face Trainer.

# 1) Instantiate tokenizer
#    - Use AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True).
#    - Keep the same tokenizer instance for all runs to ensure consistency.
#    - Verify special tokens exist (CLS/SEP/PAD); fast tokenizer is preferred.

# 2) Define a tokenization factory
#    - Write make_tokenize_fn(max_len) -> callable.
#    - The returned function should accept a batch (dict with "text") and call
#      tokenizer(..., truncation=True, max_length=max_len, padding=False).
#      (Padding=False because we will rely on a collator for dynamic padding.)
#    - IMPORTANT: Do NOT hard-code max_len; pass it as the function argument
#      so you can easily sweep different max_seq_length values from your EDA.

# 3) Build a data collator for dynamic padding
#    - Use DataCollatorWithPadding(tokenizer=tokenizer).
#    - This ensures padding is computed per-batch at runtime, minimizing waste.

# 4) Prepare tokenized datasets for a given max_len
#    - Implement prepare_datasets_for_max_len(raw_ds, max_len):
#        a) Create tokenize_fn = make_tokenize_fn(max_len).
#        b) Map over all splits with batched=True to speed up:
#             raw_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
#           (Remove original "text" to avoid duplicates; keep labels.)
#        c) Rename "label" -> "labels" to match Trainer expectations.
#        d) Set tensor format for PyTorch:
#             set_format(type="torch", columns=["input_ids","attention_mask","labels"])
#        e) Return the resulting DatasetDict (train/validation/test).



# Deliverables for this step
#    - A tokenizer instance, a make_tokenize_fn factory, a DataCollatorWithPadding,
#      and a helper that returns tensor-ready tokenized DatasetDict for any max_len.


## Task 3: Standard Fine-Tuning

- **Implement** fine-tuning for BERT with a predefined hyperparameter range (e.g., `max_length`, learning rate, batch size, epochs; exact choices are up to you). You may include additional hyperparameters based on your implementation, and there is no restriction on the search space (you may even build a custom classifier head on top of BERT).
- **Model selection:** choose the **best checkpoint** according to **validation** performance.
- **Logging:** save training logs and **plot the training loss curve**.
- **Report:** evaluate the selected checkpoint on the **test** split, reporting **accuracy** and **macro-F1**, and **plot the confusion matrix**.


In [ ]:
# =========================================================
# Procedure: Metrics + Hyperparameter Search + Summaries
# =========================================================
# What you do in this cell (implementation style is up to you:
# hand-written training loop OR Hugging Face Trainer are both fine).
#
# 1) Define eval metrics
#    - compute_metrics(eval_pred):
#        * from (logits, labels) -> preds = argmax(logits, -1)
#        * report: {"accuracy": ..., "macro_f1": ...}
#
# 2) Specify a SMALL grid (justify values from your EDA)
#    - batch_sizes: e.g., {16, 32}
#    - learning_rates: e.g., {2e-5, 3e-5, 5e-5}
#    - max_seq_lengths: e.g., {64, 96, 128}
#    - num_epochs: up to your compute resources (e.g., 3)
#    - Selection criterion: VALIDATION ACCURACY
#
# 3) For each max_seq_length:
#    - Tokenize the dataset with that max_len (reuse your prep helper).
#
# 4) For each (batch_size, learning_rate) in the grid:
#    - Initialize a fresh model (label mappings set).
#    - Train on train split; evaluate on validation split.
#    - Log/collect metrics per run (val accuracy, val macro-F1, basic history).
#    - Keep track of best run by validation accuracy; also store its trainer/state.
#
# 5) After the grid finishes:
#    - Print a compact leaderboard (top-N by val accuracy; include macro-F1).
#    - Report the best configuration (max_len, batch_size, lr, epochs).
#
# 6) Plot curves for the BEST run only (per-epoch):
#    - Training loss
#    - Validation loss
#    - Validation accuracy
#    - Validation macro-F1
#
# 7) Final test evaluation (primary: Accuracy; also report Macro-F1 & CM)
#    - Choose evaluator:the best-by-validation checkpoint.
#    - Use the tokenized TEST split aligned with the chosen max_seq_length.
#    - Run evaluation to obtain metrics dict; explicitly log PRIMARY: test accuracy.
#    - Also compute/report Macro-F1 for completeness.
#    - Generate confusion matrix with class order matching label_names; plot it.
#    - Print a per-class classification report (precision/recall/F1/support).



## Task 4: Error Analysis

**Goal:** Show ≥3 misclassified examples *per class*, include the **predicted** vs. **gold** labels, and add a brief **diagnosis** explaining why the model likely failed.

**Recommended workflow (reference only):**
1. Build a **keyword dictionary** to annotate cues in each tweet (e.g., sarcasm markers, negation, contrast).
2. For the test split, collect instances where `prediction ≠ gold`.
3. For each **class**, present at least **three** examples with:
   - **Text** (original tweet)
   - **Predicted** vs. **Gold** label
   - **Diagnosis** (short justification using detected cues)

**Example keyword dictionary (customize as needed):**
```python
sarcasm_markers = [
    "yeah right", "as if", "sure jan", "great job", "love that for me",
    "what could go wrong", "nice one", "so fun", "totally", "obviously",
]
irony_hashtags = ["#irony", "#sarcasm", "#sarcastic", "#totally", "#yeahright"]
negation_markers = [" not ", "n't ", " never ", " no ", " hardly ", " scarcely ", " without "]
contrast_markers = [" but ", " however", " though", " yet "]
```
**Suggested reporting format (per class):**

**Class:** non_irony (or irony)

**Example 1**  
**Text:** “...”  
**Pred → Gold:** non_irony → irony  
**Diagnosis:** contains sarcasm marker (“yeah right”) and contrast (“but”), likely misread.

**Example 2**  
**Text:** “...”  
**Pred → Gold:** …  
**Diagnosis:** negation and rhetorical phrasing; sentiment is ambiguous.

**Example 3**  
**Text:** “...”  
**Pred → Gold:** …  
**Diagnosis:** hashtag “#sarcastic” suggests irony; model failed to leverage it.

**Notes:**
- You may design your own dictionaries or import third-party lexicons.
- Keep diagnoses concise (one to two sentences).
- State some actionable takeaways after diagonisis.
- Optionally add automatic tags (e.g., `tags = {"sarcasm": True, "negation": False, ...}`) to support your discussion.


In [ ]:
# 8) Error Analysis (irony-focused) — Procedure (no code)
# -------------------------------------------------------
# Goal: Show ≥3 misclassified examples per GOLD class with a brief diagnosis
#       highlighting irony cues. Keep it concise and evidence-based.

# 1) Collect errors
#    - From test predictions, get indices where y_pred != y_true.
#    - Keep raw texts for those indices.

# 2) Lightweight diagnostics (heuristics; reference only, you should implemnt yours)
#    - For each text, flag presence of cues (True/False or tags):
#        * sarcasm markers (e.g., "yeah right", "as if", "great job")
#        * irony hashtags (#sarcasm, #irony, #sarcastic)
#        * negation (" not ", "n't ", "never", "no")
#        * contrast (" but ", "however", "though", "yet")
#        * quotation cues (“ ” ' ")
#        * rhetorical question (?) / phrases ("who would have thought")
#        * hyperbole/intensifiers ("literally", "absolutely", "best/worst day ever")
#        * emojis suggesting irony (🙄 😒 😏 😂 🤣)
#        * suspicious positive hashtags in negative context (#awesome, #perfect)
#    - If none matched and text is very short, tag as "short/context-dependent".

# 3) Group & sample
#    - Group misclassified examples by GOLD label (non_irony, irony).
#    - For each GOLD class, show at least 3 examples (or as many as available).

# 4) Report format (per example)
#    - Text: "…"
#    - Pred → Gold: <pred_label> → <gold_label>
#    - Diagnosis: short reason using the detected tags (e.g., "sarcasm phrase + contrast").

# 5) Tie-back
#    - Briefly state 1–2 actionable takeaways.


## Task 5: Fine-tuning with Limited Training Data

You now have access to **only 1% of the training data**. Complete the following:

### 5.1 Reproduce baseline with 1% data
- Train using **exactly 1%** of the original training split with  **updated optimal hyperparameters**, based on your previous defined hyperparameter range. Follow the same hyperparameter selection procedure.
- Briefly compare against the full-data setting (performance gap + key observations).

### 5.2 Improve performance with 1% training data
Explore techniques to boost performance under the 1% regime. There is **no restriction** on the method; you may change model families, objectives, or inference strategies. Examples include (not exhaustive):

- **Data augmentation (simple):** synonym/emoji substitutions, back-translation, paraphrasing, template-based augmentation, or class-balanced resampling.
- **Prompt-based / in-context learning (advanced):** reformulate the task as few-shot prompting with instruction templates and exemplars; optionally apply lightweight calibration or verbalizer tuning.
- **Model switching or adaptation:** replace BERT with a stronger **pretrained** encoder (e.g., RoBERTa, DeBERTa) or call **LLMs** (e.g., DeepSeek, Gemini) via official APIs; consider parameter-efficient tuning (LoRA/PEFT), layer freezing, or adapters.

> **Important:** These example approaches are suggestions only. **We do not guarantee any of them will work.** You should assess feasibility and think carefully before you try them.

> **Grading note:** If your improved method performs **worse** than the naive 1% baseline (5.1) or fails to meet the specified accuracy threshold in the marking criteria, **no marks** will be awarded.

### Practical constraints & responsibility
- All approaches depend on **your available compute and budget**.
- Some LLM providers offer **free-tier** access—verify current limits before use.
- You are responsible for **API usage**, **costs**, and compliance with provider policies.

### Deliverables
- Result comparison of fine-tuned pre-trained model under challenging and standard settings, associated with its concise discussion.
- A brief **method description**.
- A **performance report** including test **Accuracy**, **Macro-F1**, and a **confusion matrix**.
- A description on **why the gains can occur**.

In [ ]:
# Task 5.1 — Naive baseline (EXACTLY 1% of train)
# ---------------------------------------------------
# Goal: Reproduce the baseline under a 1% data regime using the SAME selection procedure as before.

# 1) Data subset
#    - From the tokenized TRAIN split (for each max_seq_length), sample EXACTLY 1% with a fixed seed.

# 2) Grid & selection (same as earlier)
#    - Sweep your small grid over (batch_size, learning_rate, max_seq_length, epochs).
#    - Train on the 1% subset; validate on the full VALIDATION split.
#    - Select BEST by VALIDATION ACCURACY.

# 3) Test evaluation (reporting)
#    - Evaluate the best 1% model on TEST.
#    - Report: Test Accuracy (primary), Macro-F1, Confusion Matrix, per-class report.

# 4) Comparison
#    - Briefly compare against the full-data setting: performance gap + 1–2 observations (e.g., stability, class-wise errors).

# 5) Documentation
#    - Log best config (max_len, bs, lr, epochs), seed, and the number of training examples used (exact count at 1%).


In [ ]:
# Task 5.2 — Peroformance improvement (on the SAME 1% regime)
# -------------------------------------------------------
# Goal: Implement ONE method to improve performance under 1% data and evaluate fairly.

# 1) Choose ONE approach
#    - Example categories: simple data augmentation, PEFT/LoRA, stronger encoder (e.g., RoBERTa/DeBERTa), or prompt/in-context.
#    - State your method in 2–3 sentences (what you change vs. baseline).

# 2) Train & select
#    - Keep the 1% training budget or less.
#    - If you resort to LLMs, you can skip training but conduct predictions in zero-shot manner. You can also choose to tune the LLM.


# 3) Test evaluation (same metrics)
#    - Evaluate on TEST and report: Accuracy, Macro-F1, Confusion Matrix.

# 4) Compare to 5.1
#    - Present a short side-by-side comparison with the 1% baseline.
#    - Give a concise reason why your method can yield gains (one paragraph max).
